In [0]:
CATALOG = "workspace"   # change if yours differs
path = f"/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv"

raw = (spark.read
       .option("header", True)
       .option("multiLine", True)     # narratives contain line breaks
       .option("quote", '"')
       .option("escape", '"')         # narratives contain quotes
       .csv(path))

raw.printSchema()
display(raw.limit(5))

root
 |-- Date received: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Sub-product: string (nullable = true)
 |-- Issue: string (nullable = true)
 |-- Sub-issue: string (nullable = true)
 |-- Company public response: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP code: string (nullable = true)
 |-- Submitted via: string (nullable = true)
 |-- Date sent to company: string (nullable = true)
 |-- Company response to consumer: string (nullable = true)
 |-- Timely response?: string (nullable = true)
 |-- Complaint ID: string (nullable = true)



Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11841177
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,Experian Information Solutions Inc.,NC,28209,Web,2025-01-29,Closed with explanation,Yes,11841190
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,null,"EQUIFAX, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11839301
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11838309
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,Experian Information Solutions Inc.,NC,28209,Web,2025-01-29,Closed with explanation,Yes,11840793


In [0]:
import re
from pyspark.sql import functions as F

def to_snake(name: str) -> str:
    return re.sub(r"[^0-9a-zA-Z]+", "_", name.strip().lower()).strip("_")

bronze_df = raw.toDF(*[to_snake(c) for c in raw.columns])   # rename only, keep all values as strings

bronze_df = (bronze_df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

(bronze_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.cfpb_bronze.complaints_raw"))

In [0]:
display(bronze_df)

date_received,product,sub_product,issue,sub_issue,company_public_response,company,state,zip_code,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id,_ingested_at,_source_file
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11841177,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,Experian Information Solutions Inc.,NC,28209,Web,2025-01-29,Closed with explanation,Yes,11841190,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,null,"EQUIFAX, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11839301,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11838309,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,Experian Information Solutions Inc.,NC,28209,Web,2025-01-29,Closed with explanation,Yes,11840793,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,null,"EQUIFAX, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11839205,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,null,"EQUIFAX, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11840971,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-01-29,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consumer and the CFPB and chooses not to provide a public response,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",NC,28209,Web,2025-01-29,Closed with non-monetary relief,Yes,11840195,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-02-26,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,null,"EQUIFAX, INC.",NC,28209,Web,2025-02-26,Closed with non-monetary relief,Yes,12234644,2026-10-27T13:11:31.509Z,dbfs:/Volumes/workspace/cfpb_bronze/raw_files/filtered_complaints.csv
2025-02-26,Credit reporting or other personal consumer reports,Credit reporting,Incorrect information on your report,Information belongs to someone else,Company has responded to the consum

In [0]:
spark.sql(f"COMMENT ON TABLE {CATALOG}.cfpb_bronze.complaints_raw IS 'Raw CFPB consumer complaints. All strings. Source: CFPB Consumer Complaint Database CSV export.'")

print("rows:", spark.table(f"{CATALOG}.cfpb_bronze.complaints_raw").count())
display(spark.sql(f"""
  SELECT count(*) AS total_rows,
         count(DISTINCT complaint_id) AS distinct_ids,
         sum(CASE WHEN complaint_id IS NULL THEN 1 ELSE 0 END) AS null_ids,
         min(date_received) AS min_date,
         max(date_received) AS max_date
  FROM {CATALOG}.cfpb_bronze.complaints_raw"""))
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.cfpb_bronze.complaints_raw"))

rows: 10298630


total_rows,distinct_ids,null_ids,min_date,max_date
10298630,10298630,0,2025-01-01,2026-10-27


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,58f37148-0383-47e9-841e-49baa5b097e4,workspace.cfpb_bronze.complaints_raw,Raw CFPB consumer complaints. All strings. Source: CFPB Consumer Complaint Database CSV export.,,2026-10-27T12:25:53.890Z,2026-10-27T13:11:35.000Z,List(),List(),1,72587592,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> dc33b291-d9c4-49b6-9982-bc2a16311c0b, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-a439bbea-cbd1-4159-90a9-1c0ec1624a6f, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-836957df-70ea-4eb0-95b0-45954fcdb1d6)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
bronze = spark.table(f"{CATALOG}.cfpb_bronze.complaints_raw")
cols = set(bronze.columns)

def col_or_null(name, alias=None):
    """Return the column if it exists, otherwise a NULL string column."""
    c = F.col(name) if name in cols else F.lit(None).cast("string")
    return c.alias(alias or name)

# If your narrative column has a different name, add it to this list
narrative_candidates = [n for n in ["consumer_complaint_narrative", "complaint_what_happened"] if n in cols]
narrative_col = F.col(narrative_candidates[0]) if narrative_candidates else F.lit(None).cast("string")

typed = bronze.select(
    F.trim(F.col("complaint_id")).alias("complaint_id"),
    F.expr("try_cast(date_received as date)").alias("date_received"),
    F.expr("try_cast(date_sent_to_company as date)").alias("date_sent_to_company"),
    F.trim(F.col("product")).alias("product"),
    F.trim(col_or_null("sub_product")).alias("sub_product"),
    F.trim(F.col("issue")).alias("issue"),
    F.trim(col_or_null("sub_issue")).alias("sub_issue"),
    narrative_col.alias("narrative"),
    F.trim(col_or_null("company_public_response")).alias("company_public_response"),
    F.trim(F.col("company")).alias("company"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.trim(col_or_null("zip_code")).alias("zip_code"),
    F.trim(col_or_null("tags")).alias("tags"),
    F.trim(col_or_null("consumer_consent_provided")).alias("consumer_consent_provided"),
    F.trim(col_or_null("submitted_via")).alias("submitted_via"),
    F.trim(F.col("company_response_to_consumer")).alias("company_response"),
    (F.trim(F.col("timely_response")) == "Yes").alias("is_timely"),
    F.col("_ingested_at"),
)

In [0]:
enriched = (typed
    .withColumn("received_month", F.date_trunc("month", "date_received").cast("date"))
    .withColumn("days_to_company_response", F.datediff("date_sent_to_company", "date_received"))
    .withColumn("got_relief", F.col("company_response").isin(
        "Closed with monetary relief", "Closed with non-monetary relief"))
    .withColumn("has_narrative",
        F.col("narrative").isNotNull() & (F.length(F.trim("narrative")) > 0)))

In [0]:
dedup = enriched.dropDuplicates(["complaint_id"])

In [0]:
flagged = dedup.withColumn("dq_reason", F.concat_ws("; ",
    F.when(F.col("complaint_id").isNull() | (F.col("complaint_id") == ""), F.lit("missing_complaint_id")),
    F.when(F.col("date_received").isNull(), F.lit("invalid_date_received")),
    F.when(F.col("date_received") > F.current_date(), F.lit("future_date_received")),
    F.when(F.col("date_sent_to_company") < F.col("date_received"), F.lit("sent_before_received")),
    F.when(F.col("company").isNull() | (F.col("company") == ""), F.lit("missing_company")),
    F.when(F.col("state").isNotNull() & (F.length("state") != 2), F.lit("invalid_state")),
))

quarantine = flagged.filter(F.col("dq_reason") != "")
silver = flagged.filter(F.col("dq_reason") == "").drop("dq_reason")

In [0]:
(silver.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.cfpb_silver.complaints"))

(quarantine.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.cfpb_silver.complaints_quarantine"))

In [0]:
b = spark.table(f"{CATALOG}.cfpb_bronze.complaints_raw").count()
d = dedup.count()
s = spark.table(f"{CATALOG}.cfpb_silver.complaints").count()
q = spark.table(f"{CATALOG}.cfpb_silver.complaints_quarantine").count()

print(f"bronze rows          : {b}")
print(f"duplicates removed   : {b - d}")
print(f"quarantined          : {q}")
print(f"silver rows          : {s}")
print(f"reconciles           : {b == (b - d) + q + s}")
display(spark.table(f"{CATALOG}.cfpb_silver.complaints_quarantine")
        .groupBy("dq_reason").count())

bronze rows          : 10298630
duplicates removed   : 0
quarantined          : 810
silver rows          : 10297820
reconciles           : True


dq_reason,count
invalid_state,810


In [0]:
spark.sql(f"COMMENT ON TABLE {CATALOG}.cfpb_silver.complaints IS 'Cleaned and typed CFPB complaints, deduplicated on complaint_id; rows failing quality rules are in complaints_quarantine'")
spark.sql(f"COMMENT ON COLUMN {CATALOG}.cfpb_silver.complaints.is_timely IS 'True if company responded within the required timeframe'")
spark.sql(f"COMMENT ON COLUMN {CATALOG}.cfpb_silver.complaints.got_relief IS 'True if closed with monetary or non-monetary relief'")
spark.sql(f"COMMENT ON COLUMN {CATALOG}.cfpb_silver.complaints.days_to_company_response IS 'Days between date received and date sent to company'")
spark.sql(f"COMMENT ON COLUMN {CATALOG}.cfpb_silver.complaints.narrative IS 'Consumer-written free text, present only if consent given; treat as sensitive'")

DataFrame[]